In [42]:
import cv2
import os
import glob
import numpy as np
from collections import defaultdict

# Configuración de rutas y parámetros
query_image_path = "../data/test_images/my_messy_500_peso_photo.jpg"
database_dir = "../data/database/sift_database/"

# JUSTIFICACIÓN DEL PARÁMETRO RATIO_THRESH = 0.7
# Este valor define la rigurosidad del "Ratio Test".
# Matemáticamente, exige que la distancia Euclidiana al mejor match sea al menos un 30% más corta que la 
# distancia al segundo mejor match (ratio < 0.7). 
# Lowe determinó empíricamente que este valor elimina el 90% de los matches falsos, perdiendo solo un 5%
# de los correctos. Es un filtro más estricto que el 0.75 usado en ORB.
RATIO_THRESH = 0.7

# JUSTIFICACIÓN DEL PARÁMETRO MIN_MATCHES = 10
# Este parámetro establece el umbral mínimo de características robustas necesarias para declarar un "reconocimiento positivo".
# Se necesitan al menos 4 puntos para calcular una matriz de homografía, pero 4 genera demasiados falsos positivos.
# Usamos 10 para garantizar empíricamente que el agrupamiento geométrico corresponde genuinamente al billete.
MIN_MATCHES = 10

def recognize_banknote(query_path, db_dir):
    print(f"Analizando: {os.path.basename(query_path)}")
    
    # Carga de imagen y detección de características
    query_img = cv2.imread(query_path)
    if query_img is None:
        print("Error: No se pudo cargar la imagen.")
        return
        
    gray_query = cv2.cvtColor(query_img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create()
    kp_query, des_query = sift.detectAndCompute(gray_query, None)
    
    if des_query is None:
        print("No se encontraron descriptores en la imagen.")
        return

    # BFMatcher con NORM_L2 para descriptores de punto flotante (SIFT)
    # Inicializamos el comparador por Fuerza Bruta debido a que el dataset referencial es manejable en tamaño.
    # Parámetro normType=cv2.NORM_L2: ES CRÍTICO cambiar esto respecto a ORB. SIFT no usa vectores binarios, 
    # sino vectores de 128 dimensiones reales. NORM_L2 calcula la distancia Euclidiana tradicional entre estos vectores, 
    # mientras que NORM_HAMMING fallaría por completo.
    # Parámetro crossCheck=False: Se desactiva para poder obtener los 2 vecinos más cercanos (k=2) en el paso KNN.
    matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    category_votes = defaultdict(int)
    db_files = glob.glob(os.path.join(db_dir, "*.npy"))
    
    # Comparación con la base de datos
    for db_file in db_files:
        des_db = np.load(db_file)
        filename = os.path.basename(db_file)
        category = filename.split("_comp_")[0].replace("norm_clean_", "")

        # JUSTIFICACIÓN DEL MÉTODO KNN Y EL PARÁMETRO k=2
        # Se utiliza knnMatch() en lugar de un match por umbral de distancia absoluta porque SIFT, al trabajar 
        # en espacios de 128 dimensiones, puede tener distancias globales muy variables dependiendo del contraste.
        # KNN garantiza encontrar los vecinos topológicamente más cercanos.
        # Utilizamos k=2 para poder aplicar el Ratio Test de Lowe, comparando matemáticamente al mejor candidato con
        # su "competidor" más cercano.
        matches = matcher.knnMatch(des_query, des_db, k=2)
        
        # Conteo de coincidencias mediante Ratio Test de Lowe
        # Comparamos la distancia euclidiana del mejor match (m_n[0]) frente al segundo mejor (m_n[1]). 
        # Si el ratio es menor a 0.7, significa que el descriptor de consulta encontró una correspondencia 
        # única y altamente distintiva en la base de datos, rechazando matches ambiguos en zonas repetitivas.
        good_matches = sum(1 for m_n in matches 
                           if len(m_n) == 2 and m_n[0].distance < RATIO_THRESH * m_n[1].distance)

        if good_matches >= MIN_MATCHES:
            category_votes[category] += good_matches 

    # Visualización de resultados
    if not category_votes:
        print("Resultado: Billete no reconocido.")
    else:
        results = sorted(category_votes.items(), key=lambda x: x[1], reverse=True)
        winner, score = results[0]
        
        print("-" * 40)
        print(f"BILLETE DETECTADO: {winner}")
        print(f"Puntos de coincidencia: {score}")
        print("-" * 40)

        i = 0
        for winner, score in results: 
            print(f"Candidato {i}: {winner} ({score} matches)")
            i += 1
            
       # if len(results) > 1:
       #     print(f"Candidato secundario: {results[1][0]} ({results[1][1]} matches)")

recognize_banknote(query_image_path, database_dir)

----------------------------------------
Analizando: my_messy_500_peso_photo.jpg
----------------------------------------
BILLETE DETECTADO: 500PesosBack
Puntos de coincidencia: 284
----------------------------------------
Candidato 0: 500PesosBack (284 matches)
Candidato 1: 50PesosPolimeroBack (98 matches)
Candidato 2: 20PesosBack (98 matches)
Candidato 3: 1000PesosBack (90 matches)
Candidato 4: 500PesosFront (81 matches)
Candidato 5: 1000PesosFront (65 matches)
Candidato 6: 20PesosFront (50 matches)
Candidato 7: 50PesosPolimeroFront (49 matches)
Candidato 8: 100PesosBack (45 matches)
Candidato 9: 20PesosPolimeroFront (45 matches)
Candidato 10: 200PesosBack (45 matches)
Candidato 11: 200PesosFront (40 matches)
Candidato 12: 20PesosPolimeroBack (21 matches)
Candidato 13: 100PesosFront (13 matches)
